# Implementing empiric survey data in a model

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/18-empiric-contact-model.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Now let's feed these empiric data into a model. The easiest case is modelling a
population that is the same as, or similar to, the one surveyed; then it is
reasonable to use the matrix as it is. POLYMOD was carried out from late 2005
through much of 2006, so if we use the Great Britain matrix to simulate an
infection in that population we should be fine — provided we also use that
population's age structure.

We start from the same matrix as {doc}`17-empiric-surveys`, together with the
age structure of the United Kingdom in 2006, the population data the source
textbook uses.

The matrix is the POLYMOD Great Britain matrix from Mossong *et al.* (2008),
supporting table
[doi:10.1371/journal.pmed.0050074.st005](https://doi.org/10.1371/journal.pmed.0050074.st005)
(Creative Commons Attribution licence), as transcribed in the source textbook.
summer4 ships no survey data: this page embeds the extract it uses, so nothing
is downloaded when the documentation builds.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ContactMatrix, ForceOfInfection

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# POLYMOD Great Britain, as published: rows are the AGE OF CONTACT and columns
# the AGE GROUP OF PARTICIPANT.
PUBLISHED = np.array(
    [
        [1.92, 0.65, 0.41, 0.24, 0.46, 0.73, 0.67, 0.83, 0.24, 0.22, 0.36, 0.20, 0.20, 0.26, 0.13],
        [0.95, 6.64, 1.09, 0.73, 0.61, 0.75, 0.95, 1.39, 0.90, 0.16, 0.30, 0.22, 0.50, 0.48, 0.20],
        [0.48, 1.31, 6.85, 1.52, 0.27, 0.31, 0.48, 0.76, 1.00, 0.69, 0.32, 0.44, 0.27, 0.41, 0.33],
        [0.33, 0.34, 1.03, 6.71, 1.58, 0.73, 0.42, 0.56, 0.85, 1.16, 0.70, 0.30, 0.20, 0.48, 0.63],
        [0.45, 0.30, 0.22, 0.93, 2.59, 1.49, 0.75, 0.63, 0.77, 0.87, 0.88, 0.61, 0.53, 0.37, 0.33],
        [0.79, 0.66, 0.44, 0.74, 1.29, 1.83, 0.97, 0.71, 0.74, 0.85, 0.88, 0.87, 0.67, 0.74, 0.33],
        [0.97, 1.07, 0.62, 0.50, 0.88, 1.19, 1.67, 0.89, 1.02, 0.91, 0.92, 0.61, 0.76, 0.63, 0.27],
        [1.02, 0.98, 1.26, 1.09, 0.76, 0.95, 1.53, 1.50, 1.32, 1.09, 0.83, 0.69, 1.02, 0.96, 0.20],
        [0.55, 1.00, 1.14, 0.94, 0.73, 0.88, 0.82, 1.23, 1.35, 1.27, 0.89, 0.67, 0.94, 0.81, 0.80],
        [0.29, 0.54, 0.57, 0.77, 0.97, 0.93, 0.57, 0.80, 1.32, 1.87, 0.61, 0.80, 0.61, 0.59, 0.57],
        [0.33, 0.38, 0.40, 0.41, 0.44, 0.85, 0.60, 0.61, 0.71, 0.95, 0.74, 1.06, 0.59, 0.56, 0.57],
        [0.31, 0.21, 0.25, 0.33, 0.39, 0.53, 0.68, 0.53, 0.55, 0.51, 0.82, 1.17, 0.85, 0.85, 0.33],
        [0.26, 0.25, 0.19, 0.24, 0.19, 0.34, 0.40, 0.39, 0.47, 0.55, 0.41, 0.78, 0.65, 0.85, 0.57],
        [0.09, 0.11, 0.12, 0.20, 0.19, 0.22, 0.13, 0.30, 0.23, 0.13, 0.21, 0.28, 0.36, 0.70, 0.60],
        [0.14, 0.15, 0.21, 0.10, 0.24, 0.17, 0.15, 0.41, 0.50, 0.71, 0.53, 0.76, 0.47, 0.74, 1.47],
    ]
)
AGE_GROUPS = list(range(0, 75, 5))  # 0-4, 5-9, ..., 65-69, 70+
BANDS = (*AGE_GROUPS, math.inf)

UK_2006 = np.array(
    [3458060, 3556024, 3824317, 3960916, 3911291, 3762213, 4174675, 4695853,
     4653082, 3986098, 3620216, 3892985, 3124676, 2706365, 6961183],
    dtype=float,
)

age_pops = pd.Series(UK_2006, index=pd.Index(AGE_GROUPS, name="age"))
print(f"Total population in the age data is {age_pops.sum() / 1e6:.1f} million")
age_pops.plot.area(
    title="United Kingdom population by age, 2006",
    labels={"index": "age", "value": "population"},
)

The model is the SIR of the previous chapters with frequency-dependent
transmission, stratified by an `age` property whose traits are the lower bound
of each band (`"0"`, `"5"`, …, `"70"`). The contact matrix is aligned to that
property by position, and because the trait names are numbers summer4 also
checks that they are the band edges. `to_mixing()` gives the force of infection
the survey's own contact numbers — no row normalisation — so the flow's
parameter is the risk of transmission per contact.

The total population is split across age groups in proportion to the UK age
structure, and so is the seed of infectious people.

In [ ]:
state = Property("state", ("susceptible", "infectious", "recovered"))
age = Property("age", tuple(str(lower) for lower in AGE_GROUPS))
contacts = ContactMatrix.from_array(PUBLISHED.T, BANDS, prop=age, source_population=UK_2006)

model_config = {"end_time": 40.0, "population": 60e6, "seed": 100.0}
parameters = {"risk_per_contact": 0.1, "infectious_period": 4.0}

pmap = PropertyMap.from_property(state).stratify(age)
model = FlowModel(pmap)
infection = ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=age,
    mixing=contacts.to_mixing(),
    contact_rate=Param("risk_per_contact"),
)
model.add_flow(TransitionFlow("infection", state["susceptible"], state["infectious"], infection))
model.add_flow(
    TransitionFlow(
        "recovery", state["infectious"], state["recovered"], 1.0 / Param("infectious_period")
    )
)
uk_model = model.compile()

split = UK_2006 / UK_2006.sum()
y0 = np.zeros(pmap.size)
for trait, share in zip(age.traits, split):
    y0[pmap.select(state["susceptible"] & age[trait])] = share * (
        model_config["population"] - model_config["seed"]
    )
    y0[pmap.select(state["infectious"] & age[trait])] = share * model_config["seed"]

Set some arbitrary parameters for an imaginary infection in this population
(above), run the model, and compute prevalence — the proportion of each age
group currently infectious.

In [ ]:
days = np.arange(0.0, model_config["end_time"] + 1.0)
plan = SavePlan(requests={"infectious": SaveRequest(Compartments(where=state["infectious"]), ts=days)})
result = uk_model.run(
    parameters, y0, t0=0.0, t1=model_config["end_time"], dt=0.1, save=plan, solver="euler"
)
group_sizes = split * model_config["population"]
prevalence = pd.DataFrame(
    np.asarray(result["infectious"].values.data) / group_sizes,
    index=pd.Index(days, name="time"),
    columns=[f"{lower:02d}" for lower in AGE_GROUPS],
)
prevalence.iloc[[10, 20, 30]].round(4)

We probably do not expect dramatic age effects: the only difference between
the age groups is their contact rates, and no other epidemiological difference
has been put in. Even so, heterogeneous mixing should produce some small
differences between the groups.

In [ ]:
window = prevalence.iloc[10:30].transpose()
fig = go.Figure(data=[go.Contour(x=window.columns, y=window.index, z=window.values)])
fig.update_xaxes(title="time")
fig.update_yaxes(title="age group")
fig.update_traces(colorbar_title_text="prevalence")
fig.update_layout(title="Prevalence by age group over time")
fig

In [ ]:
fig = go.Figure(data=[go.Surface(x=window.columns, y=window.index, z=window.values)])
fig.update_layout(
    title="Prevalence by age group over time, as a surface",
    width=800,
    height=600,
    scene=dict(xaxis_title="time", yaxis_title="age group", zaxis_title="prevalence"),
)
fig

Transmission is a little more intense, and happens a little earlier, in the
younger age groups: the school-age groups peak earlier and higher than the
oldest.

In [ ]:
peak_day = prevalence.idxmax()
peak_height = prevalence.max()
assert peak_day["10"] <= peak_day["70"]
assert peak_height["10"] > peak_height["70"]
pd.DataFrame({"peak day": peak_day, "peak prevalence": peak_height.round(4)}).T